## **Laboratorio Clase 1: Explorando la API de un LLM**

Usaremos noticias tecnologicas como hilo conductor para explorar como funciona la API de OpenAI: 
desde como el modelo ve el texto hasta tecnicas de prompt engineering.

### Que practicaras

- Tokenizacion: como convierte el modelo el texto en numeros
- Parametros de generacion (`temperature`, `max_output_tokens`) con casos reales
- Conversacion multi-turno con historial
- Prompt engineering: system prompts, few-shot, chain-of-thought

### Objetivos

Al finalizar seras capaz de:

- Conectar con la API y hacer llamadas basicas
- Visualizar como el modelo transforma texto en tokens numericos
- Controlar la creatividad y la longitud de las respuestas con parametros
- Construir un chatbot que recuerda el contexto de la conversacion
- Aplicar tecnicas de prompt engineering para tareas NLP reales

## Seccion 1: Configuracion del entorno


**Celda 1: Instalacion de librerias**


In [1]:
!pip install --upgrade openai tiktoken


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


`openai`: libreria oficial de OpenAI. `tiktoken`: el tokenizador de OpenAI, nos permite ver exactamente como el modelo convierte texto en tokens numericos — ilustrando que los modelos *no entienden palabras, entienden numeros*.

**Celda 2: Clave de API**


In [ ]:
OPENAI_API_KEY = "sk-..."  # Reemplaza con tu clave real

La clave de API identifica tu cuenta. Nunca la compartas ni la subas a repositorios publicos.

**Celda 3: Cliente y funcion auxiliar**


In [4]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

def preguntar(pregunta, temperatura=0.7, max_tokens=300):
    r = client.responses.create(
        model='gpt-4o-mini',
        input=pregunta,
        temperature=temperatura,
        max_output_tokens=max_tokens
    )
    return r.output_text

print('Cliente listo. Probando...')
print(preguntar('Di hola en 5 palabras.', temperatura=0.0))

Cliente listo. Probando...
¡Hola! ¿Cómo estás hoy?


Creamos `preguntar()` con temperatura y longitud configurables. La probamos con una llamada minima.

## Seccion 2: Tokenizacion — Como ve el modelo el texto


Las diapositivas mostraron que los algoritmos no entienden palabras, entienden numeros. Aqui lo verificamos directamente.

**Celda 4: Contar tokens en distintos textos**

In [5]:
import tiktoken

enc = tiktoken.encoding_for_model('gpt-4o-mini')

textos = [
    'Hola',
    'inteligencia artificial',
    'Los LLMs no entienden palabras, entienden tokens.',
    'The quick brown fox jumps over the lazy dog.'
]

print(f"{'Texto':<52} {'Tokens':>6}  IDs")
print('-' * 80)
for texto in textos:
    ids = enc.encode(texto)
    ids_str = str(ids[:5]) + '...' if len(ids) > 5 else str(ids)
    print(f"{texto[:50]:<52} {len(ids):>6}  {ids_str}")

Texto                                                Tokens  IDs
--------------------------------------------------------------------------------
Hola                                                      1  [49864]
inteligencia artificial                                   4  [491, 15570, 5174, 27034]
Los LLMs no entienden palabras, entienden tokens.        13  [16593, 451, 19641, 82, 860]...
The quick brown fox jumps over the lazy dog.             10  [976, 4853, 19705, 68347, 65613]...


**Observaciones clave:**
- El espanol suele usar mas tokens que el ingles para el mismo significado (mayor coste)
- Los tokens son subpalabras, no palabras completas
- Lo que el modelo 've' son estos numeros, no las letras

**Celda 5: Decodificar — ver las 'piezas' de cada token**


In [6]:
# Visualizar exactamente que fragmento representa cada token
frase = 'inteligencia artificial'
tokens = enc.encode(frase)

print(f'Frase: "{frase}"')
print(f'IDs numericos: {tokens}')
print()
print('Cada token decodificado:')
for t in tokens:
    fragmento = enc.decode([t])
    print(f'  ID {t:6d}  ->  "{fragmento}"')

print()
print('--- Comparacion espanol vs ingles (mismo significado) ---')
pares = [('casa', 'house'), ('electroencefalografista', 'electroencephalographer'), ('modelo', 'model')]
for es, en in pares:
    n_es = len(enc.encode(es))
    n_en = len(enc.encode(en))
    print(f'{es:25s} {n_es} tok  |  {en:25s} {n_en} tok')

Frase: "inteligencia artificial"
IDs numericos: [491, 15570, 5174, 27034]

Cada token decodificado:
  ID    491  ->  "int"
  ID  15570  ->  "elig"
  ID   5174  ->  "encia"
  ID  27034  ->  " artificial"

--- Comparacion espanol vs ingles (mismo significado) ---
casa                      2 tok  |  house                     1 tok
electroencefalografista   6 tok  |  electroencephalographer   5 tok
modelo                    1 tok  |  model                     1 tok


Esto explica por que los prompts en espanol cuestan mas que en ingles: mas tokens = mas coste. Optimizar la longitud de los prompts es una practica de produccion importante.

## Seccion 3: Parametros de Generacion


Usaremos noticias tecnologicas para ver como los parametros cambian el comportamiento.

**Celda 6: Efecto de `temperature`**

In [8]:
noticia = (
    'Apple ha anunciado su nuevo chip M4 Ultra con Neural Engine de 32 nucleos, '
    'capaz de procesar 40 billones de operaciones por segundo y ejecutar modelos '
    'de 70B parametros directamente en el dispositivo, sin servidores externos.'
)

print('=' * 65)
print('TEMPERATURE 0.0 — preciso, siempre igual')
print('=' * 65)
for i in range(2):
    r = client.responses.create(model='gpt-4o-mini', input=f'Resume en 1 frase: {noticia}', temperature=0.0)
    print(f'Intento {i+1}: {r.output_text.strip()}')

print()
print('=' * 65)
print('TEMPERATURE 1.3 — creativo, cada vez diferente')
print('=' * 65)
for i in range(2):
    r = client.responses.create(model='gpt-4o-mini', input=f'Escribe un titular impactante para: {noticia}', temperature=1.3)
    print(f'Intento {i+1}: {r.output_text.strip()}')

TEMPERATURE 0.0 — preciso, siempre igual
Intento 1: Apple ha presentado el chip M4 Ultra, que cuenta con un Neural Engine de 32 núcleos y puede procesar 40 billones de operaciones por segundo, permitiendo ejecutar modelos de 70 mil millones de parámetros directamente en el dispositivo sin necesidad de servidores externos.
Intento 2: Apple ha presentado el chip M4 Ultra, que cuenta con un Neural Engine de 32 núcleos y puede procesar 40 billones de operaciones por segundo, permitiendo ejecutar modelos de 70 mil millones de parámetros directamente en el dispositivo.

TEMPERATURE 1.3 — creativo, cada vez diferente
Intento 1: **¡Revolución Apple! El nuevo chip M4 Ultra con 32 núcleos y 40 billones de operaciones por segundo transforma la computación en dispositivos sin depender de servidores externos.**
Intento 2: "Revolución tecnológica: Apple lanza el M4 Ultra, el poder de 40 billones de operaciones por segundo en la palma de tu mano."


`temperature=0.0` es determinista: el mismo prompt produce siempre la misma respuesta. `temperature=1.3` explora tokens menos probables: respuestas mas creativas pero variables. Para tareas analiticas (clasificacion, extraccion) usa 0.0. Para escritura creativa, valores mas altos.

**Celda 7: Efecto de `max_output_tokens` — del tweet al parrafo**


In [ ]:
print('=== 25 tokens — tweet ===')
r = client.responses.create(model='gpt-4o-mini', input=f'Resume para Twitter (muy breve): {noticia}', max_output_tokens=25)
print(r.output_text)
print(f'Tokens usados: {r.usage.output_tokens}')

print()
print('=== 150 tokens — parrafo ===')
r = client.responses.create(model='gpt-4o-mini', input=f'Explica para un blog tecnico: {noticia}', max_output_tokens=150)
print(r.output_text)
print(f'Tokens usados: {r.usage.output_tokens}')

=== 25 tokens — tweet ===
🚀 Apple presenta el chip M4 Ultra con Neural Engine de 32 núcleos, capaz de procesar 40
Tokens usados: 25

=== 150 tokens — parrafo ===
# Apple Lanza el Chip M4 Ultra: Poder de Procesamiento en la Palma de tu Mano

Apple ha dado un paso significativo en el mundo del procesamiento de datos con el lanzamiento de su nuevo chip M4 Ultra. Este potente procesador, que incluye un Neural Engine de 32 núcleos, está diseñado para llevar el rendimiento de los dispositivos a nuevas alturas, al tiempo que mejora la eficiencia energética y la capacidad de procesamiento en el dispositivo.

## Potencia de Procesamiento Impactante

Con una capacidad de realizar **40 billones de operaciones por segundo**, el M4 Ultra redefine lo que es posible para los dispositivos móviles y de escritorio. Este impresionante rendimiento no solo mejora la velocidad de ejecución de tareas cotidianas, sino que también permite ejecutar aplicaciones complejas, como modelos de inteligencia artificial

`max_output_tokens` controla la longitud maxima de la respuesta. El modelo se corta bruscamente al llegar al limite — planifica un margen. Clave para gestionar costes y adaptar el formato al canal.

## Seccion 4: Conversacion Multi-Turno


El modelo **no recuerda** por defecto. Para construir un chatbot con memoria debemos enviar el historial completo en cada llamada.

**Celda 8: Chatbot con historial**

In [11]:
historial = [
    {'role': 'system', 'content': 'Eres un asistente experto en tecnologia e IA. Responde en 2-3 frases maximo.'}
]

def chat(mensaje):
    historial.append({'role': 'user', 'content': mensaje})
    r = client.responses.create(model='gpt-4o-mini', input=historial)
    respuesta = r.output_text
    historial.append({'role': 'assistant', 'content': respuesta})
    return respuesta

preguntas = [
    '¿Que es un chip Neural Engine?',
    '¿Y cuantos parametros pueden tener los modelos que ejecuta?',
    '¿Que ventaja tiene ejecutarlos localmente frente a en la nube?'
]

for p in preguntas:
    print(f'Usuario:    {p}')
    print(f'Asistente:  {chat(p)}')
    print()

Usuario:    ¿Que es un chip Neural Engine?
Asistente:  Un Neural Engine es un componente de hardware diseñado específicamente para ejecutar algoritmos de inteligencia artificial y aprendizaje automático de manera eficiente. Se utiliza en dispositivos como smartphones y computadoras para tareas como reconocimiento de voz, procesamiento de imágenes y otras aplicaciones que requieren procesamiento intensivo de datos.

Usuario:    ¿Y cuantos parametros pueden tener los modelos que ejecuta?
Asistente:  Los modelos que ejecuta un Neural Engine pueden tener desde cientos de miles hasta miles de millones de parámetros, dependiendo de la complejidad del modelo. Por ejemplo, algunos modelos modernos de deep learning, como los de lenguaje natural o visión por computadora, pueden superar los 175 mil millones de parámetros. 

Usuario:    ¿Que ventaja tiene ejecutarlos localmente frente a en la nube?
Asistente:  Ejecutar modelos localmente ofrece ventajas como menor latencia, ya que elimina el tiemp

La **tercera pregunta** no especifica de que, y el modelo responde correctamente porque el historial completo se envia en cada llamada. El contexto es la clave del chatbot.

**Celda 9: Ver el historial que ve el modelo**


In [12]:
print('=== HISTORIAL ENVIADO AL MODELO EN CADA LLAMADA ===')
for msg in historial:
    rol = msg['role'].upper()
    texto = msg['content'][:90] + '...' if len(msg['content']) > 90 else msg['content']
    print(f'[{rol}]\n  {texto}')
    print()
print(f'Total: {len(historial)} mensajes enviados en cada peticion')

=== HISTORIAL ENVIADO AL MODELO EN CADA LLAMADA ===
[SYSTEM]
  Eres un asistente experto en tecnologia e IA. Responde en 2-3 frases maximo.

[USER]
  ¿Que es un chip Neural Engine?

[ASSISTANT]
  Un Neural Engine es un componente de hardware diseñado específicamente para ejecutar algor...

[USER]
  ¿Y cuantos parametros pueden tener los modelos que ejecuta?

[ASSISTANT]
  Los modelos que ejecuta un Neural Engine pueden tener desde cientos de miles hasta miles d...

[USER]
  ¿Que ventaja tiene ejecutarlos localmente frente a en la nube?

[ASSISTANT]
  Ejecutar modelos localmente ofrece ventajas como menor latencia, ya que elimina el tiempo ...

Total: 7 mensajes enviados en cada peticion


El historial crece con cada turno: mas tokens de entrada = mayor coste. En produccion se usa ventana deslizante (descartar mensajes viejos) o compresion del historial.

## Seccion 5: Ingenieria de Prompts


**Celda 10: System prompt — periodista tecnologica**


In [13]:
r = client.responses.create(
    model='gpt-4o-mini',
    input=[
        {
            'role': 'system',
            'content': (
                'Eres Elena Vargas, periodista tecnologica de El Pais con 15 anos de experiencia. '
                'Escribes con estilo directo usando analogias para no-tecnicos. '
                'Siempre terminas con una reflexion sobre el impacto social. Maximo 3 parrafos.'
            )
        },
        {
            'role': 'user',
            'content': 'Explica que es un transformer en IA y por que fue revolucionario.'
        }
    ]
)
print(r.output_text)

Un transformer en inteligencia artificial es como un maestro de orquesta en el mundo del procesamiento del lenguaje. Imagina que cada instrumento (o palabra) tiene un papel que desempeñar en una melodía (un texto). Los transformers son capaces de escuchar todo el conjunto a la vez, en lugar de seguir una secuencia lineal. Esto les permite entender mejor el contexto y las relaciones entre las palabras, lo que resulta en un aprendizaje mucho más profundo y preciso.

La revolución que trajo consigo esta tecnología radica en su capacidad para manejar grandes volúmenes de datos y aprender de ellos de manera autónoma. Antes de los transformers, los modelos de IA eran como un traductor que siempre seguía el mismo camino, pero ahora son como un intérprete que puede improvisar, captar matices y aportar creatividad. Esto ha llevado a avances significativos en tareas como la traducción automática, la generación de texto y el análisis de sentimientos, entre otros.

En un mundo donde la comunicació

El **system prompt** define la persona, el estilo y las restricciones del modelo para toda la sesion. En aplicaciones reales contiene instrucciones permanentes, restricciones de seguridad y contexto del dominio.

**Celda 11: Few-shot prompting — clasificador de noticias**


In [14]:
def clasificar(titular):
    r = client.responses.create(
        model='gpt-4o-mini',
        input=[
            {'role': 'system',    'content': 'Clasifica titulares en: tecnologia, politica, economia, ciencia, deportes. Solo la categoria.'},
            {'role': 'user',      'content': 'Tesla presenta Optimus, su robot humanoide para fabricas'},
            {'role': 'assistant', 'content': 'tecnologia'},
            {'role': 'user',      'content': 'El BCE sube los tipos al 4.5% para frenar la inflacion'},
            {'role': 'assistant', 'content': 'economia'},
            {'role': 'user',      'content': 'El Congreso aprueba la nueva ley de presupuestos'},
            {'role': 'assistant', 'content': 'politica'},
            {'role': 'user',      'content': 'La ESA detecta agua liquida en la luna Europa de Jupiter'},
            {'role': 'assistant', 'content': 'ciencia'},
            {'role': 'user',      'content': titular}
        ],
        temperature=0.0
    )
    return r.output_text.strip()

titulares_test = [
    'OpenAI lanza GPT-5 con capacidades de razonamiento avanzado',
    'El Real Madrid gana la Champions por decimoquinta vez',
    'La inflacion en Espana baja al 2.1% en mayo',
    'Investigadores del CERN descubren una nueva particula subatomica',
    'El gobierno aprueba el plan de pensiones para 2027',
    'Amazon despide al 8% de su plantilla en recorte global'
]

print('Clasificacion de titulares:')
for t in titulares_test:
    cat = clasificar(t)
    print(f'  [{cat.upper():12s}]  {t}')

Clasificacion de titulares:
  [TECNOLOGIA  ]  OpenAI lanza GPT-5 con capacidades de razonamiento avanzado
  [DEPORTES    ]  El Real Madrid gana la Champions por decimoquinta vez
  [ECONOMIA    ]  La inflacion en Espana baja al 2.1% en mayo
  [CIENCIA     ]  Investigadores del CERN descubren una nueva particula subatomica
  [POLITICA    ]  El gobierno aprueba el plan de pensiones para 2027
  [ECONOMIA    ]  Amazon despide al 8% de su plantilla en recorte global


El **few-shot prompting** guia al modelo con ejemplos sin reentrenamiento. Con 4 ejemplos tenemos un clasificador de 5 categorias bastante preciso. El ultimo titular es ambiguo (tecnologia/economia) — los casos limite revelan las limitaciones del modelo.

**Celda 12: Chain-of-thought — verificar si una noticia es verosimil**


In [15]:
titulares_dudosos = [
    'Un estudiante de 14 anos crea una IA que diagnostica cancer con 99.9% de precision usando selfies',
    'Google DeepMind anuncia AGI certificada para finales de 2025'
]

for titular in titulares_dudosos:
    print(f'Titular: "{titular}"')
    print('-' * 65)
    r = client.responses.create(
        model='gpt-4o-mini',
        input=(
            f'Analiza si este titular es verosimil, exagerado o probablemente falso.\n'
            f'Razona PASO A PASO antes del veredicto final:\n\n'
            f'1. ¿Que afirma exactamente?\n'
            f'2. ¿Que conocimiento tecnico/cientifico se necesita para evaluarlo?\n'
            f'3. ¿Hay inconsistencias o exageraciones?\n'
            f'4. VEREDICTO: (verosimil / exagerado / probablemente falso) + 1 frase.\n\n'
            f'Titular: "{titular}"'
        )
    )
    print(r.output_text)
    print()

Titular: "Un estudiante de 14 anos crea una IA que diagnostica cancer con 99.9% de precision usando selfies"
-----------------------------------------------------------------
### Análisis del Titular

1. **¿Qué afirma exactamente?**
   - El titular afirma que un estudiante de 14 años ha desarrollado una inteligencia artificial (IA) capaz de diagnosticar cáncer con un 99.9% de precisión utilizando selfies. Esto implica que la IA puede analizar imágenes faciales y detectar la enfermedad con una eficacia excepcional.

2. **¿Qué conocimiento técnico/científico se necesita para evaluarlo?**
   - Para evaluar esta afirmación, se requiere un entendimiento en áreas como:
     - **Inteligencia Artificial y Aprendizaje Automático**: Cómo se diseñan, entrenan y evalúan modelos de IA, especialmente en el ámbito médico.
     - **Diagnóstico Médico**: Conocer los métodos utilizados para diagnosticar cáncer y las herramientas que generalmente se aplican, como bioquímica, radiología y análisis clínico

El **chain-of-thought** mejora drasticamente la precision en tareas logicas. Al pedir que el modelo 'piense en voz alta' antes de responder, reduce alucinaciones y errores de razonamiento. Especialmente util para verificacion de hechos, analisis y resolucion de problemas complejos.